In [1]:
from pathlib import Path
import ims
from skimage.io import imshow
from sympy.strategies.branch import condition

import preprocess

# Step 0: Define all necessary directories.
input_dir = Path("./data/IMS_Olive_Oil_Geography_small")
output_dir = Path("./output/Jupyter_Test/")

# Check and create directories if necessary.
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

# Load the test dataset and interpolate to a common RIP
dataset = ims.Dataset.read_mea(path=input_dir,
                               subfolders=True,
                               sequential=True,
                               compr="binning")

# Get the first test spectrum.
spectrum = dataset[0]

In [14]:
spectrum.meta_attr

['Chunks count             = 0000006939',
 'Chunk averages           = 6',
 'Chunk sample count       = 3150',
 'Chunk sample rate        = 150 [kHz]',
 'Chunk trigger duration   = 150 [µs]',
 'Chunk trigger repetition = 21 [ms]',
 'Chunk voltrange          = 10 [V]',
 'Class                    = "manual/pos"',
 'Machine serial           = "OEM-100005"',
 'Timestamp                = "2015-12-07T14:46:57"',
 'OEM Software Version     = "01.02"',
 'Machine type             = "OEM Module"',
 'Sample                   = 8']

In [18]:
import numpy as np

peak_window_ret_time = spectrum.ret_time[500:510]
peak_window_drift_time = spectrum.drift_time[500:510]

R_matrix, D_matrix = np.meshgrid(peak_window_ret_time, peak_window_drift_time, indexing='ij')
peak_window_complete = np.stack((R_matrix, D_matrix), axis=-1)

peak_window_complete

array([[[73.5       ,  0.478317  ],
        [73.5       ,  0.47927364],
        [73.5       ,  0.48023027],
        [73.5       ,  0.48118691],
        [73.5       ,  0.48214354],
        [73.5       ,  0.48310017],
        [73.5       ,  0.48405681],
        [73.5       ,  0.48501344],
        [73.5       ,  0.48597008],
        [73.5       ,  0.48692671]],

       [[73.647     ,  0.478317  ],
        [73.647     ,  0.47927364],
        [73.647     ,  0.48023027],
        [73.647     ,  0.48118691],
        [73.647     ,  0.48214354],
        [73.647     ,  0.48310017],
        [73.647     ,  0.48405681],
        [73.647     ,  0.48501344],
        [73.647     ,  0.48597008],
        [73.647     ,  0.48692671]],

       [[73.794     ,  0.478317  ],
        [73.794     ,  0.47927364],
        [73.794     ,  0.48023027],
        [73.794     ,  0.48118691],
        [73.794     ,  0.48214354],
        [73.794     ,  0.48310017],
        [73.794     ,  0.48405681],
        [73.794     ,  0

In [7]:
# =========================================== JUST PLOTTING THE CUTTED RAW SPECTRUM ===========================================
import plot
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw_cut")


<Figure size 1000x600 with 0 Axes>

In [2]:
# =========================================== I. PREPROCESSING ===========================================
import plot
from scipy.ndimage import uniform_filter
# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw",
                       peak_cords=None)

spectrum.rip_ms = 1
# Baseline correction - removing the noisy background by subtracting the mean of an empty area.
spectrum.sub_first_rows(50)
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

#uniform_filter_size= (11, 11)
#spectrum.values = uniform_filter(input=spectrum.values, size=uniform_filter_size)
#from scipy.ndimage import gaussian_filter
#spectrum.values = gaussian_filter(spectrum.values, sigma=1.0)

# Smoothing the spectrum with the Savitzky–Golay filter.
spectrum.savgol(window_length=10, polyorder=4)

# Remove negative values after applying the smooth filter.
spectrum.values[spectrum.values < 0] = 0.0

# Plot the processed spectrum.

plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="processed",
                       peak_cords=None)


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [ ]:
import numpy as np
np.percentile(spectrum.values, [99.5])

# Inspect some tree parameters here ...
print(f"SPECTRUM INFOS")
print(f"Type: {type(spectrum.values)}")
print(f"Element Type: {type(spectrum.values[0, 0])}")
print("================================================")
print(f"Shape: {spectrum.values.shape}")
num_entries = spectrum.values.shape[0] * spectrum.values.shape[1]
num_non_0_entries = len(np.flatnonzero(spectrum.values))
print(f"Total Entries: {num_entries}")
print(f"Non-0 Enties: {num_non_0_entries}")
print(f"0 Enties: {num_entries - num_non_0_entries}")
print("================================================")
print(f"Max Value: {spectrum.values.max()}")
print(f"Min Value: {spectrum.values.min()}")
print("================================================")
over_50_values = len(np.where(non_normalized_spectrum.values >= 25)[0])
print(f"Total Values: {num_entries}")
print(f"Values > 50: {over_50_values}")
print(f"Values > 50 in % {(over_50_values/num_entries)*100}")
print(f"Percentile Value: {100 - (over_50_values/num_entries)*100}")



In [19]:
import numpy as np
from pathlib import Path

file = Path("./output/test/npy/151210_192049/PW_151210_192049_RT63_DT364_21x21.npy")

npy_file = np.load(file)


In [36]:
npy_file[0,0]

array([ 4.08367292, 88.494     ,  1.36320346])

In [18]:
import os
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from matplotlib import pyplot as plt
import networkx as nx
from scipy.spatial import KDTree
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


def process_peaks_from_excel(excel_file: Path,
                             output_dir: Path,
                             filter_types: list[str],
                             threshold: float = 0.001) -> pd.DataFrame:

    # Check if the Excel file exists.
    if not excel_file.exists():
        print(f"File not found: {excel_file}")
        return

    # Load the Excel file.
    peak_dataframe = pd.read_excel(excel_file)
    # Filter entries by type.
    if filter_types:
        peak_dataframe = peak_dataframe[peak_dataframe['label'].isin(filter_types)].reset_index(drop=True)

    # Check if the Excel file has all required columns.
    required_cols = {'ret_time', 'drift_time', 'label'}
    if not required_cols.issubset(peak_dataframe.columns):
        raise ValueError(f"Missing columns. Required: {required_cols}")

    # Create connected components.
    process_peaks_and_plot(peak_dataframe=peak_dataframe, output_dir=output_dir, threshold=threshold)

    print("Completed - calculations and plots")
    return peak_dataframe


def process_peaks_and_plot(peak_dataframe: pd.DataFrame,
                           output_dir: Path,
                           threshold: float) -> None:

    # Calculate connected components.
    graph, components, peak_dataframe = calculate_connected_components(peak_dataframe,
                                                                       output_dir,
                                                                       threshold,
                                                                       batch_size=20000)

    """
    #############################################################################
    # # Skalierung der Retentions- und Driftzeiten für spätere Zuweisung
    scaler = MinMaxScaler()
    scaled_times = scaler.fit_transform(peak_dataframe[['ret_time', 'drift_time']].values)

    # Berechnet die Honigtyp-Anteile pro Komponente
    honey_stats = count_honey_type_proportions(components, peak_dataframe, scaled_times)

    # Plottet und speichert das Ergebnis der Zusammenhangskomponenten
    plot_connected_components(graph, components, output_dir)
    komponente = components[109]  # oder eine andere Komponente

    plot_einzene_komponente(graph, komponente, output_dir, filename="komponente_drift_retention.png")

    get_average_peaks(graph, peak_dataframe, components, output_dir, filename="scatter_peaks_with_components")

    # Berechnet die Durchschnittspunkte für jede Komponente
    avg_peaks = get_average_peaks(graph, peak_dataframe, components, output_dir, filename="scatter_peaks_with_components")

    # Extrahiert die Honiganteile für jede Komponente
    component_honey_counts = [(honey['Linde_Count'], honey['Lavendel_Count'], honey['Eukalyptus_Count']) for honey in
                              honey_stats.values()]

    # Plottet Durchschnittspunkte mit den gemischten Farben basierend auf den Honiganteilen
    plot_peaks_by_honey_type(avg_peaks, component_honey_counts, output_dir, filename_prefix="peaks_by_honey_type")
    """

# Funktion: baut einen Graphen aus den Peak-Daten und bildet Zusammenhangskomponenten
def calculate_connected_components(peak_dataframe: pd.DataFrame,
                                   output_dir: Path,
                                   threshold: float,
                                   batch_size: int):

     # Initialize a graph.
    graph = nx.Graph()

    # TODO -> Each peak entry consists of 121 different neighbor peaks. Each of these peaks must be added as a node.

    # Combine the retention and drifttimes into one 2D array and normalize the values [0-1].
    # Shape: [[R1, D1], [R2, D2], .... , [Rn, Dn]]
    scaler = MinMaxScaler()
    scaled_times = scaler.fit_transform(peak_dataframe[['ret_time', 'drift_time']].values)

    print(type(scaled_times))
    print(scaled_times.shape)

    # Each peak is added as a node in the graph.
    for idx in range(len(peak_dataframe)):
        node = tuple(scaled_times[idx])
        peak_data = peak_dataframe.iloc[idx]
        graph.add_node(node_for_adding=node, label=peak_data['label'], npy_file=peak_data['npy_file'], dataframe_idx=idx)

    # Create a nearest neighbor tree for fast lookup of neighbor nodes.
    tree = KDTree(scaled_times)
    # Number of nodes in the graph.
    node_count = graph.number_of_nodes()

    # Create edges between all nodes within a defined radius. Create batches of nodes and walk through them.
    # neighbor_nodes is a list with lists inside. The position of the list is the index of the node.
    # The inner list contains the positions of nodes which are considered the nodes neighbors.
    for start_node in range(0, node_count, batch_size):
        end_node = min(start_node + batch_size, node_count)
        sub_nodes = scaled_times[start_node:end_node]
        neighbor_nodes = tree.query_ball_point(sub_nodes, r=threshold, p=2)

        for sub_node_idx, neighbors in enumerate(neighbor_nodes):
            global_idx = start_node + sub_node_idx
            node_1 = tuple(scaled_times[global_idx])
            for neighbor_idx in neighbors:
                if global_idx != neighbor_idx:
                    node_2 = tuple(scaled_times[neighbor_idx])
                    graph.add_edge(node_1, node_2)


    # Sucht alle Zusammenhangskomponenten
    # Jede Komponente ist eine Menge von Knoten, die über Kanten verbunden sind
    connected_components = list(nx.connected_components(graph))
    # Erstellt ein Dictionary, das jedem Peak-Index seine component_id zuordnet.
    component_mapping = {}
    for component_idx, component in enumerate(connected_components):
        for node in component:
            dataframe_idx = graph.nodes[node]['dataframe_idx']
            component_mapping[dataframe_idx] = component_idx
    # Fügt die component_id in den ursprünglichen DataFrame ein.
    peak_dataframe['component_idx'] = peak_dataframe.index.map(component_mapping)
    # Baut eine Liste mit allen Peaks + ihren Attributen
    # Verwendet skalierte Koordinaten (node[0], node[1]).
    peak_component_list = []
    for component_idx, component in enumerate(connected_components):
        for node in component:
            node_data = graph.nodes[node]
            peak_component_list.append({
                'index': node_data['dataframe_idx'],
                'label': node_data['label'],
                'ret_time': node[0],
                'drift_time': node[1],
                'npy_file': node_data['npy_file'],
                'component_idx': component_idx
            })

    peak_component_dataframe = pd.DataFrame(peak_component_list)

    csv_filename = output_dir / "peak_components.csv"
    peak_component_dataframe.to_csv(csv_filename, index=False)
    # Save peak date to Excel file.
    excel_filename = output_dir / "peak_components.xlsx"
    peak_component_dataframe.to_excel(excel_filename, index=False)

    return graph, connected_components, peak_dataframe

In [19]:
from pathlib import Path

output_dir = Path("./output/test")
excel_filename = output_dir / "All_peak_data.xlsx"

process_peaks_from_excel(excel_file=excel_filename,
                         output_dir=output_dir,
                         threshold=0.002,
                         filter_types=["Greece", "Italy", "Spain", "test"])

<class 'numpy.ndarray'>
(36, 2)
Completed - calculations and plots


,label,spectrum,ret_time,drift_time,intensity,npy_file,component_idx
0,test,151210_192049,89.229,1.367987,71.001672,output\test\npy\151210_192049\PW_151210_192049...,0
1,test,151210_192049,104.811,1.050384,678.305861,output\test\npy\151210_192049\PW_151210_192049...,1
2,test,151210_192049,104.664,1.177616,304.605425,output\test\npy\151210_192049\PW_151210_192049...,2
3,test,151210_192049,116.277,1.240754,102.604007,output\test\npy\151210_192049\PW_151210_192049...,3
4,test,151210_192049,116.277,1.330678,95.151996,output\test\npy\151210_192049\PW_151210_192049...,4
5,test,151210_192049,116.571,1.113522,357.495302,output\test\npy\151210_192049\PW_151210_192049...,5
6,test,151210_192049,116.571,1.155614,68.559252,output\test\npy\151210_192049\PW_151210_192049...,6
7,test,151210_192049,125.979,1.051341,112.839236,output\test\npy\151210_192049\PW_151210_192049...,7
8,test,151210_192049,132.153,1.510525,171.529595,output\test\npy\151210_192049\PW_151210_192049...,8
9,test,151210_192049,149.205,1.367987,111.507931,output\test\npy\151210_192049\PW_151210_192049...,9


NameError: name 'scaled_times' is not defined